## makemore: becoming a backprop ninja

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
%matplotlib inline

In [2]:
# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

zsh:1: command not found: wget


In [4]:
# read in all the words
words = open('data/name.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [5]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [9]:
# build the dataset
block_size = 3  # context length: how many characters do we take to predict the next one?


def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])  # 10%

torch.Size([182441, 3]) torch.Size([182441])
torch.Size([22902, 3]) torch.Size([22902])
torch.Size([22803, 3]) torch.Size([22803])


In [ ]:
# ok biolerplate done, now we get to the action:

In [11]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [12]:
n_embd = 10  # the dimensionality of the character embedding vectors
n_hidden = 64  # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)  # for reproducibility
embedding_table = torch.randn((vocab_size, n_embd), generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / ((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
# BatchNorm parameters
batch_norm_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
batch_norm_bias = torch.randn((1, n_hidden)) * 0.1

parameters = [embedding_table, W1, b1, W2, b2, batch_norm_gain, batch_norm_bias]
print(sum(p.nelement() for p in parameters))  # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [17]:
batch_size = 32
n = batch_size  # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # batch X,Y

torch.Size([182441, 3])


In [149]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

embeddings = embedding_table[Xb]  # embed the characters into vectors
embeddings_concat = embeddings.view(embeddings.shape[0], -1)  # concatenate the vectors

# Linear layer 1
hidden_layer_pre_batch_norm = embeddings_concat @ W1 + b1  # hidden layer pre-activation

# BatchNorm layer
batch_norm_mean_i = 1 / n * hidden_layer_pre_batch_norm.sum(0, keepdim=True)
batch_norm_diff = hidden_layer_pre_batch_norm - batch_norm_mean_i
batch_norm_diff_2 = batch_norm_diff ** 2
batch_norm_var = 1 / (n - 1) * (batch_norm_diff_2).sum(0, keepdim=True)  # note: Bessel's correction (dividing by n-1, not n)
batch_norm_var_inverse = (batch_norm_var + 1e-5) ** -0.5
batch_norm_raw = batch_norm_diff * batch_norm_var_inverse
hpreact = batch_norm_gain * batch_norm_raw + batch_norm_bias

# Non-linearity
hidden_layer = torch.tanh(hpreact)  # hidden layer

# Linear layer 2
logits = hidden_layer @ W2 + b2  # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes  # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1  # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,  # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, hidden_layer, hpreact, batch_norm_raw,
          batch_norm_var_inverse, batch_norm_var, batch_norm_diff_2, batch_norm_diff, hidden_layer_pre_batch_norm,
          batch_norm_mean_i,
          embeddings_concat, embeddings]:
    t.retain_grad()
loss.backward()
loss

tensor(3.4989, grad_fn=<NegBackward0>)

In [158]:
hidden_layer_pre_batch_norm.shape, embeddings_concat.shape, W1.shape, b1.shape

(torch.Size([32, 64]),
 torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([64]))

In [165]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one
# [ 2, 3, 2 ] [2]
# [ 2, 4, 5]  [5]

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1 / n
cmp('logprobs', dlogprobs, logprobs)

dprobs = 1 / probs * dlogprobs
cmp('probs', dprobs, probs)

dcounts_sum_inv = (counts * dprobs).sum(dim=1, keepdim=True)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum = - 1 / counts_sum ** 2 * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

dcounts = torch.ones_like(counts) * dcounts_sum + (counts_sum_inv * dprobs)
cmp('counts', dcounts, counts)

dnorm_logits = norm_logits.exp() * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

dlogit_maxes = (-torch.ones_like(dnorm_logits) * dnorm_logits).sum(dim=1, keepdim=True)
cmp('logit_maxes', dlogit_maxes, logit_maxes)

dlogits = torch.zeros_like(logits)
dlogits[range(n), logits.max(1).indices] = 1
dlogits = dlogits * dlogit_maxes + dnorm_logits.clone()
cmp('logits', dlogits, logits)

dhidden_layer = dlogits @ W2.T
cmp('h', dhidden_layer, hidden_layer)

dW2 = hidden_layer.T @ dlogits
cmp('W2', dW2, W2)

db2 = dlogits.sum(dim=0)
cmp('b2', db2, b2)

dhpreact = dhidden_layer * (1 - hidden_layer ** 2)
cmp('hpreact', dhpreact, hpreact)

dbatch_norm_gain = (dhpreact * batch_norm_raw).sum(dim=0, keepdim=True)
cmp('bngain', dbatch_norm_gain, batch_norm_gain)

dbatch_norm_bias = (dhpreact * torch.ones_like(batch_norm_bias)).sum(dim=0, keepdim=True)
cmp('bnbias', dbatch_norm_bias, batch_norm_bias)

dbatch_norm_raw = dhpreact * batch_norm_gain
cmp('bnraw', dbatch_norm_raw, batch_norm_raw)

dbatch_norm_var_inverse = (dbatch_norm_raw * batch_norm_diff).sum(dim=0, keepdim=True)
cmp('bnvar_inv', dbatch_norm_var_inverse, batch_norm_var_inverse)

dbatch_norm_var = dbatch_norm_var_inverse * (-0.5 / (batch_norm_var + 1e-5) ** 1.5)
cmp('bnvar', dbatch_norm_var, batch_norm_var)

dbatch_norm_diff_2 = dbatch_norm_var * (1/(n-1))
cmp('bndiff2', dbatch_norm_diff_2, batch_norm_diff_2)

dbatch_norm_diff = (dbatch_norm_diff_2 * 2*batch_norm_diff) +  (batch_norm_var_inverse * dbatch_norm_raw)
cmp('bndiff', dbatch_norm_diff, batch_norm_diff)

dbatch_norm_mean_i =  (dbatch_norm_diff * -1).sum(dim=0, keepdim=True)
cmp('bnmeani', dbatch_norm_mean_i, batch_norm_mean_i)

dhidden_layer_pre_batch_norm = (dbatch_norm_mean_i * (1/n)).sum(dim=0, keepdim=True) + dbatch_norm_diff
cmp('hprebn', dhidden_layer_pre_batch_norm, hidden_layer_pre_batch_norm)

dembeddings_concat = dhidden_layer_pre_batch_norm @ W1.T
cmp('embcat', dembeddings_concat, embeddings_concat)

dW1 = embeddings_concat.T @ dhidden_layer_pre_batch_norm
cmp('W1', dW1, W1)

db1 = dhidden_layer_pre_batch_norm.sum(0, keepdim=True)
cmp('b1', db1, b1)

dembeddings = dembeddings_concat.view(embeddings.shape)
cmp('emb', dembeddings, embeddings)

dembeddings_table = torch.zeros_like(embedding_table)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dembeddings_table[ix] += dembeddings[k, j]

cmp('C', dembeddings_table, embedding_table)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = None  # TODO. my solution is 3 lines
# -----------------

#cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

In [ ]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = batch_norm_gain * (
        hidden_layer_pre_batch_norm - hidden_layer_pre_batch_norm.mean(0, keepdim=True)) / torch.sqrt(
    hidden_layer_pre_batch_norm.var(0, keepdim=True, unbiased=True) + 1e-5) + batch_norm_bias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
dhprebn = None  # TODO. my solution is 1 (long) line
# -----------------

cmp('hprebn', dhprebn, hidden_layer_pre_batch_norm)  # I can only get approximate to be true, my maxdiff is 9e-10

In [ ]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10  # the dimensionality of the character embedding vectors
n_hidden = 200  # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)  # for reproducibility
embedding_table = torch.randn((vocab_size, n_embd), generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / ((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
# BatchNorm parameters
batch_norm_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
batch_norm_bias = torch.randn((1, n_hidden)) * 0.1

parameters = [embedding_table, W1, b1, W2, b2, batch_norm_gain, batch_norm_bias]
print(sum(p.nelement() for p in parameters))  # number of parameters in total
for p in parameters:
    p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size  # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
#with torch.no_grad():

# kick off optimization
for i in range(max_steps):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]  # batch X,Y

    # forward pass
    embeddings = embedding_table[Xb]  # embed the characters into vectors
    embeddings_concat = embeddings.view(embeddings.shape[0], -1)  # concatenate the vectors
    # Linear layer
    hidden_layer_pre_batch_norm = embeddings_concat @ W1 + b1  # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------
    bnmean = hidden_layer_pre_batch_norm.mean(0, keepdim=True)
    batch_norm_var = hidden_layer_pre_batch_norm.var(0, keepdim=True, unbiased=True)
    batch_norm_var_inverse = (batch_norm_var + 1e-5) ** -0.5
    batch_norm_raw = (hidden_layer_pre_batch_norm - bnmean) * batch_norm_var_inverse
    hpreact = batch_norm_gain * batch_norm_raw + batch_norm_bias
    # -------------------------------------------------------------
    # Non-linearity
    hidden_layer = torch.tanh(hpreact)  # hidden layer
    logits = hidden_layer @ W2 + b2  # output layer
    loss = F.cross_entropy(logits, Yb)  # loss function

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()  # use this for correctness comparisons, delete it later!

    # manual backprop! #swole_doge_meme
    # -----------------
    # YOUR CODE HERE :)
    dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -----------------

    # update
    lr = 0.1 if i < 100000 else 0.01  # step learning rate decay
    for p, grad in zip(parameters, grads):
        p.data += -lr * p.grad  # old way of cheems doge (using PyTorch grad from .backward())
        #p.data += -lr * grad # new way of swole doge TODO: enable

    # track stats
    if i % 10000 == 0:  # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    if i >= 100:  # TODO: delete early breaking when you're ready to train the full net
        break

In [ ]:
# useful for checking your gradients
# for p,g in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), g, p)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
    # pass the training set through
    embeddings = embedding_table[Xtr]
    embeddings_concat = embeddings.view(embeddings.shape[0], -1)
    hpreact = embeddings_concat @ W1 + b1
    # measure the mean/std over the entire training set
    bnmean = hpreact.mean(0, keepdim=True)
    batch_norm_var = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad()  # this decorator disables gradient tracking
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = embedding_table[x]  # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)  # concat into (N, block_size * n_embd)
    hpreact = embcat @ W1 + b1
    hpreact = batch_norm_gain * (hpreact - bnmean) * (batch_norm_var + 1e-5) ** -0.5 + batch_norm_bias
    h = torch.tanh(hpreact)  # (N, n_hidden)
    logits = h @ W2 + b2  # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())


split_loss('train')
split_loss('val')

In [ ]:
# I achieved:
# train 2.0718822479248047
# val 2.1162495613098145

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size  # initialize with all ...
    while True:
        # forward pass
        embeddings = embedding_table[torch.tensor([context])]  # (1,block_size,d)
        embeddings_concat = embeddings.view(embeddings.shape[0], -1)  # concat into (N, block_size * n_embd)
        hpreact = embeddings_concat @ W1 + b1
        hpreact = batch_norm_gain * (hpreact - bnmean) * (batch_norm_var + 1e-5) ** -0.5 + batch_norm_bias
        hidden_layer = torch.tanh(hpreact)  # (N, n_hidden)
        logits = hidden_layer @ W2 + b2  # (N, vocab_size)
        # sample
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))